# Customer Churn Forecasting - Feature Engineering
## Module 2: Data Preprocessing & Feature Transformation

This notebook handles data cleaning, encoding categorical variables, creating new features, and preparing the dataset for machine learning models.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully.')

## 2.1 Load and Clean the Dataset

In [ ]:
# Load dataset
import os
if os.path.exists('WA_Fn-UseC_-Telco-Customer-Churn.csv'):
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
elif os.path.exists('telco_customer_churn.csv'):
    df = pd.read_csv('telco_customer_churn.csv')
else:
    raise FileNotFoundError('Dataset not found.')

print(f'Original dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Step 1: Handle TotalCharges - convert to numeric and handle missing values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check missing values after conversion
missing_tc = df['TotalCharges'].isnull().sum()
print(f'Missing TotalCharges after conversion: {missing_tc}')

# Fill missing TotalCharges with median (these are typically new customers with tenure=0)
if missing_tc > 0:
    # Show the rows with missing TotalCharges
    print('\nRows with missing TotalCharges:')
    display(df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])
    
    # Fill with tenure * MonthlyCharges (logical imputation)
    df['TotalCharges'].fillna(df['tenure'] * df['MonthlyCharges'], inplace=True)
    print(f'\nMissing values filled. Remaining nulls: {df["TotalCharges"].isnull().sum()}')

In [ ]:
# Step 2: Drop customerID (non-predictive identifier)
df.drop(columns=['customerID'], inplace=True)
print(f'Dropped customerID. Shape: {df.shape}')

## 2.2 Encode Target Variable

In [ ]:
# Encode the target variable: Churn (Yes=1, No=0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(f'Target variable encoded:')
print(df['Churn'].value_counts())
print(f'\nChurn Rate: {df["Churn"].mean()*100:.2f}%')

## 2.3 Create New Features (Feature Engineering)

In [ ]:
# Feature 1: Average Monthly Spend (TotalCharges / tenure)
# Handle division by zero for tenure=0
df['AvgMonthlySpend'] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    df['MonthlyCharges']
)

# Feature 2: Tenure Groups (categorize tenure into buckets)
def tenure_group(tenure):
    if tenure <= 6:
        return '0-6 Months'
    elif tenure <= 12:
        return '6-12 Months'
    elif tenure <= 24:
        return '1-2 Years'
    elif tenure <= 48:
        return '2-4 Years'
    else:
        return '4+ Years'

df['TenureGroup'] = df['tenure'].apply(tenure_group)

# Feature 3: Total Services Subscribed
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

# Count 'Yes' values across service columns
df['TotalServices'] = 0
for col in service_cols:
    if col in df.columns:
        df['TotalServices'] += (df[col].isin(['Yes', 'Fiber optic', 'DSL'])).astype(int)

# Feature 4: Has Internet Service
df['HasInternet'] = (df['InternetService'] != 'No').astype(int)

# Feature 5: Monthly Charges per Service
df['ChargesPerService'] = np.where(
    df['TotalServices'] > 0,
    df['MonthlyCharges'] / df['TotalServices'],
    df['MonthlyCharges']
)

# Feature 6: Contract Risk Score (month-to-month is highest risk)
contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
df['ContractRisk'] = df['Contract'].map(contract_risk)

print('New features created:')
print(f'  - AvgMonthlySpend: mean={df["AvgMonthlySpend"].mean():.2f}')
print(f'  - TenureGroup: {df["TenureGroup"].value_counts().to_dict()}')
print(f'  - TotalServices: mean={df["TotalServices"].mean():.2f}')
print(f'  - HasInternet: {df["HasInternet"].value_counts().to_dict()}')
print(f'  - ChargesPerService: mean={df["ChargesPerService"].mean():.2f}')
print(f'  - ContractRisk: {df["ContractRisk"].value_counts().to_dict()}')

## 2.4 Encode Categorical Variables

In [ ]:
# Identify categorical columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns to encode ({len(cat_cols)}):')
for col in cat_cols:
    print(f'  {col}: {df[col].nunique()} unique values -> {df[col].unique()[:5]}')

In [ ]:
# Binary encoding for columns with 2 unique values
binary_cols = [col for col in cat_cols if df[col].nunique() == 2]
le = LabelEncoder()

for col in binary_cols:
    df[col] = le.fit_transform(df[col])
    print(f'  Binary encoded: {col}')

# One-hot encoding for columns with more than 2 unique values
multi_cat_cols = [col for col in cat_cols if df[col].nunique() > 2]
print(f'\nOne-hot encoding columns: {multi_cat_cols}')

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True, dtype=int)

print(f'\nDataset shape after encoding: {df.shape}')
print(f'Columns: {list(df.columns)}')

## 2.5 Feature Scaling

In [ ]:
# Scale numerical features
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 
                      'AvgMonthlySpend', 'TotalServices', 'ChargesPerService', 'ContractRisk']

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numerical_features] = scaler.fit_transform(df[numerical_features])

print('Scaled numerical features:')
for col in numerical_features:
    print(f'  {col}: mean={df_scaled[col].mean():.4f}, std={df_scaled[col].std():.4f}')

## 2.6 Visualize Engineered Features

In [ ]:
# Visualize new features vs churn
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# AvgMonthlySpend by Churn
sns.boxplot(x='Churn', y='AvgMonthlySpend', data=df, ax=axes[0, 0],
            palette={0: '#2ecc71', 1: '#e74c3c'})
axes[0, 0].set_title('Avg Monthly Spend by Churn', fontweight='bold')
axes[0, 0].set_xticklabels(['Retained', 'Churned'])

# TotalServices by Churn
sns.countplot(x='TotalServices', hue='Churn', data=df, ax=axes[0, 1],
              palette={0: '#2ecc71', 1: '#e74c3c'})
axes[0, 1].set_title('Total Services by Churn', fontweight='bold')
axes[0, 1].legend(['Retained', 'Churned'])

# ChargesPerService by Churn
sns.boxplot(x='Churn', y='ChargesPerService', data=df, ax=axes[0, 2],
            palette={0: '#2ecc71', 1: '#e74c3c'})
axes[0, 2].set_title('Charges per Service by Churn', fontweight='bold')
axes[0, 2].set_xticklabels(['Retained', 'Churned'])

# ContractRisk by Churn
sns.countplot(x='ContractRisk', hue='Churn', data=df, ax=axes[1, 0],
              palette={0: '#2ecc71', 1: '#e74c3c'})
axes[1, 0].set_title('Contract Risk Score by Churn', fontweight='bold')
axes[1, 0].set_xlabel('Risk Score (1=Low, 3=High)')
axes[1, 0].legend(['Retained', 'Churned'])

# Tenure Group by Churn
tenure_order = ['0-6 Months', '6-12 Months', '1-2 Years', '2-4 Years', '4+ Years']
if 'TenureGroup' in df.columns:
    tenure_churn = pd.crosstab(df['TenureGroup'], df['Churn'], normalize='index') * 100
    tenure_churn = tenure_churn.reindex(tenure_order)
    tenure_churn.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[1, 1].set_title('Churn Rate by Tenure Group', fontweight='bold')
    axes[1, 1].set_ylabel('Percentage (%)')
    axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=30)
    axes[1, 1].legend(['Retained', 'Churned'])

# HasInternet by Churn
if 'HasInternet' in df.columns:
    internet_churn = pd.crosstab(df['HasInternet'], df['Churn'], normalize='index') * 100
    internet_churn.index = ['No Internet', 'Has Internet']
    internet_churn.plot(kind='bar', ax=axes[1, 2], color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[1, 2].set_title('Churn by Internet Service', fontweight='bold')
    axes[1, 2].set_ylabel('Percentage (%)')
    axes[1, 2].set_xticklabels(axes[1, 2].get_xticklabels(), rotation=0)
    axes[1, 2].legend(['Retained', 'Churned'])

plt.suptitle('Engineered Features vs Churn', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('engineered_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save the processed dataset for model training
# Drop TenureGroup (already encoded via ContractRisk and tenure)
if 'TenureGroup' in df.columns:
    df.drop(columns=['TenureGroup'], inplace=True)

df.to_csv('processed_churn_data.csv', index=False)
print(f'\nProcessed dataset saved: processed_churn_data.csv')
print(f'Final shape: {df.shape}')
print(f'Final columns ({len(df.columns)}): {list(df.columns)}')

## 2.7 Feature Engineering Summary

### Transformations Applied:
1. **TotalCharges:** Converted to numeric, missing values imputed with `tenure * MonthlyCharges`
2. **customerID:** Dropped (non-predictive identifier)
3. **Target Encoding:** Churn mapped to binary (Yes=1, No=0)
4. **New Features Created:**
   - `AvgMonthlySpend`: TotalCharges / tenure (customer lifetime value indicator)
   - `TenureGroup`: Categorical tenure buckets for analysis
   - `TotalServices`: Count of active service subscriptions
   - `HasInternet`: Binary flag for internet service
   - `ChargesPerService`: Monthly charges normalized by service count
   - `ContractRisk`: Ordinal risk score based on contract type
5. **Categorical Encoding:** Binary + One-Hot encoding for all categorical features
6. **Feature Scaling:** StandardScaler applied to numerical features